# appendixB · 工程速查表

## Why this appendix
教程主线讲过概念, 但是工程中的反射动作应该是"我看到任务 X 就想到分布 Y、估计器 Z、采样器 W". 这张表把全课程 18 章浓缩成两张速查表.

## 学完后能解决
- 给一类工程任务 (回归 / 分类 / 排序 / 强化 / 排队) 立刻选出推荐分布族
- 对应估计器立刻找出 (MLE / Bayesian / 矩法)
- 采样器选对 (拒绝 / bootstrap / HMC / NUTS / reparam)
- 常见坑导航到对应章节


## 1. 场景 → 分布族决策表

| 工程场景 | 推荐 likelihood | 推荐 prior (Bayesian 时) | 章节 |
|---|---|---|---|
| 二分类 click / 留存 | Bernoulli(p) + softmax | Beta(1, 1) 或 Beta(α, β) | ch02 ch12 ch08 |
| 多分类输出 | Categorical | Dirichlet | ch03 ch11 BCE ch08 |
| n 次尝试成功数 | Bin(n, p) | Beta(α+k, β+n-k) | ch02 ch12 |
| 单位时间内事件数 | Poisson(λ) | Gamma(α, β) | ch03 ch10 ch12 |
| 服务时长 / 等待 / 间隔 | Exp(λ) | Gamma(α+, β+n) | ch04 ch10 |
| 磁盘/机器寿命 | Weibull(λ, k) (k 可变以判别 失效率走势) | 先 k=W自定义非共轭; MCMC | ch04 ch12 ch13 |
| Pixel 回归输出 | Normal(μ, σ²) σ² 常数 | Normal-Normal 共轭 | ch04 ch08 ch12 |
| 阶梯式 response集 的偏序 | Plackett-Luce | -- | ch17 RLHF |
| 上下文中的离散行动 退 | Softmax policy | Dirichlet | ch15 ch17 PPO/DPO |
| 高维隐元生成 (VAE/Diffusion latent) | N(μ_q, σ_q²) | N(0, I) | ch14 VAE |
| 长期 rate 估计 v.s 脚踝采样 | Gamma-Poisson or NegBin (overdispersion) | -- | ch03 ch06 ch12 |
| 排队 M/M/1 服务建模 | Exp arrival + Exp service | -- | ch10 |
| single observation 后验 | 多变按 conjugate; numpyro NUTS | -- | ch12 ch13 |


## 2. 场景 → 估计器决策表

| 任务 | 估计器 | 章节 |
|---|---|---|
| 大样本均值 | sample mean (LLN 大 数 律) | ch06 |
| 任意分布参数 MLE | NLL gradient descent / LBFGS | ch08 |
| 共轭贝叶斯 | closed form update | ch12 |
| Hierarchical 模型 (non-conj) | PyMC / numpyro NUTS | ch12 ch13 |
| difficulty 数 未知 | Bootstrap CI | ch07 |
| 队长 / 等待 P95 | M/M/1 analytic; 仿真核对 | ch10 |
| A/B 테스트 CTR 差异 | Beta-Binomial 共轭 + Monte-Carlo P(B>A) | ch02 ch12 |
| LLM 偏好直接学习 | DPO closed form | ch17 |
| Strategy 学习 离散动作 | REINFORCE / PPO | ch15 ch17 |
| 重参数化 latent variable 训练 | ELBO + reparam trick | ch14 |
| Feature 相关性 selection | MI / |corr|; 同时判 confounder | ch05 ch11 |


## 3. 场景 → 采样器决策表

| 场景 | 推荐采样器 | 章节 |
|---|---|---|
| 任意 1D target, 易归一化 | 拒绝采样 | ch13 |
| 任意 1D conditional density, 估计量 | Importance sampling | ch13 |
| 高维 target, gradient available | HMC / NUTS (PyMC / numpyro) | ch12 ch13 |
| 高维 target, gradient NOT available | Metropolis-Hastings / Slice | ch13 |
| 任意统计量 CI 不依赖模型 | Bootstrap | ch07 |
| 高斯 latent z (μ, σ²) 到 sample | reparameterization trick | ch14 |
| 多模 target 与 valley | tempering / parallel tempering; 或 NUTS | ch13 |
| Bandit 不 平衡反馈离线评估 | Importance reweighting | ch13 |
| 连续 latent 在 RL 中 | reparam + pathwise (reparametrized PPO) | ch14 ch15 |
| LLM 离散 token 梯度传播 | REINFORCE (score function); 或 Gumbel-softmax trick | ch15 ch17 |


## 4. 常见坑速查 (按章节)

| 误点 | 章节 | 正解 |
|---|---|---|
| 用 np.random.seed (全局单例, 多线程不安全) | ch00 | default_rng(seed) |
| 把 sample variance 当无偏 MLE (ddof=0 vs 1) | ch08 ch06 | 大样本无偏时方差用 ddof=1 |
| 相关 衍 130 因果 (Simpson 反转) | ch05 | 控 confounder 后判因果 |
| 把 LLN 当 CLT (收敛速度 unrecognized) | ch06 | LLN 给值; CLT 给分布 |
| Tail 重 -> Bootstrap 慢收回中位数 | ch07 | BCa / n 增加 |
| 在 BKI 数学 中 区分 KL forward / reverse | ch11 ch14 | VAE: forward (mass-covering); EM 反 |
| Ventation collapse β 调 0 调 错 | ch14 | free-bits / β-warmup |
| PageRank 不去 teleporting 导致黑洞 | ch09 | d=0.85 标 fix |
| M/M/1 假设错 location ρ=0.95 延迟爆炸 | ch10 | ρ ≤ 0.7-0.8 设计 |
| RLHF ε-clip ε 过大梯度 fly | ch17 | 开始 ε=0.2; 调到 0.1 |
| log p near 0 直接 np.log | appendixA | clip 或 log p 区间 |
| logits 极大 softmax 溢出 | appendixA | F.cross_entropy with logits |


## 5. 工程快查 - 一行调用

### Maximum acceleration table

| 操心 | 一行命令 |
|---|---|
| Empirical CI of median | `np.percentile(boot_medians, [2.5, 97.5])` |
| Beta posterior CrI | `from scipy.stats import beta; beta.ppf([.025, .975], α+k, β+n-k)` |
| P(B > A) Monte Carlo | `(rng.beta(α_B, β_B, N) > rng.beta(α_A, β_A, N)).mean()` |
| KL closed on Gauss | `0.5*(mu_q**2 + sigma_q**2 - 1 - log(sigma_q**2))` |
| MGF of Normal | `exp(mu*t + sigma**2 * t**2 / 2)` |
| PageRank iterate | `v = v @ (d * P + (1-d) / N * ones((N, N)))` |
| Stable softmax from logits | `F.softmax(logits, dim=-1)` (内部走 LSE) |
| Stable cross-entropy | `F.cross_entropy(logits, target)` ( BCE-with-logits variant) |


## 6. pitfalls

- 表只是地震 gesture, 真实 data 中 variance 可能 multiple 分布
- 通常要先估 用 Poisson 或 Binomial; 若 dispersion ratio 样本 var/均值 ≈ 1 Poisson; ≫ 1 过离散 NegBin / Negative Binomial Reg
- 优先用 closed form сonjugate ; 没有 experienced samples 时 fallback NUTS
- 复杂 高维梯度 (如 latent mid VAE) 一定用 reparam; 离散 action 用 REINFORCE
- 永远重启 Notebook; 受惊 时混 RunnablePath
- 总报 seed: 报几个注意 all in 仅出 现 reports `torch.manual_seed(0)` 与 `np.random.default_rng(?=0)` 一道 。


## 7. 课后 → exercises.md

## 8. 参考
全课程 18 章 + appendixA.
